In [1]:
%pip install --quiet -U python-dotenv pydantic-ai 'torchaudio<2.9' torchvision ffmpeg-python whisperx 'yt-dlp[default]'

import os
from pathlib import Path
import nest_asyncio; nest_asyncio.apply()

# Download data files if not already present (e.g. on Colab)
if not Path("data").exists():
    import zipfile, urllib.request
    url = "https://github.com/jsoma/workshop-ai-images-video/raw/main/docs/nicar-2026/05-recipes-data.zip"
    print("Downloading data...")
    urllib.request.urlretrieve(url, "_data.zip")
    with zipfile.ZipFile("_data.zip") as zf:
        zf.extractall("data")
    Path("_data.zip").unlink()
    print("Done!")

# Paste API keys here or leave empty for .env / Colab secrets
api_keys = {"OPENAI_API_KEY": "", "GOOGLE_API_KEY": "", "HF_TOKEN": ""}
os.environ.update({k: v for k, v in api_keys.items() if v})
try:
    from google.colab import userdata
    for key in api_keys:
        try: os.environ.setdefault(key, userdata.get(key))
        except Exception: pass
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

DATA = Path("data")
Path("outputs").mkdir(exist_ok=True)


/Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-nicar/01-fri-analyzing-images-videos-ai/ai-images-video/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# The full pipeline

## Let's snag some content from a meeting

Transcription isn't perfect, and summaries aren't either. But if you plug one into the other... it's definitely tempting. Let's try it with a city council meeting.


In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo("buEGUxrz8ho", width=500, height=281)


It's 2.5 hours long, which is *far too long* for us to sit through ourselves (right?). Instead, we'll see if we can get to do all the work.


**`recipes/meeting-minutes.py`** — Download a public meeting, transcribe it, summarize with Gemini.


In [2]:
import warnings, logging, os
warnings.filterwarnings("ignore")
os.dup2(os.open(os.devnull, os.O_WRONLY), 2)
logging.disable(logging.ERROR)
os.environ["TQDM_DISABLE"] = "1"


In [ ]:
from pathlib import Path
import yt_dlp
from pydantic import BaseModel, Field
from pydantic_ai import Agent

DATA = Path("data")

URL = "https://www.youtube.com/watch?v=buEGUxrz8ho"
VIDEO_ID = "buEGUxrz8ho"
MODEL = "google-gla:gemini-2.5-flash"


We'll start by downloading the audio.


In [ ]:
audio_path = DATA / f"{VIDEO_ID}.mp3"
if not audio_path.exists():
    ydl_opts = {
        "outtmpl": str(DATA / "%(id)s.%(ext)s"),
        "format": "bestaudio/best",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "mp3"}],
        "quiet": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([URL])
print(f"Audio ready: {audio_path.name}")


Audio ready: buEGUxrz8ho.mp3


Now we'll transcribe. I know we talked about Whisper this and Whisper that, but there's a new model from NVIDIA called Parakeet, and the [Parakeet MLX](https://github.com/senstella/parakeet-mlx) implementation is super super fast on macOS. I transcribed this entire 2.5 hour meeting in about a minute. In the code below it first tries Parakeet, but if you don't have it (or are on Colab or Windows) it falls back to WhisperX.


In [ ]:
try:
    from parakeet_mlx import from_pretrained
    print("Loading parakeet-mlx...")
    parakeet = from_pretrained("mlx-community/parakeet-tdt-0.6b-v3")
    print("Transcribing (chunked for long audio)...")
    result = parakeet.transcribe(str(audio_path), chunk_duration=600, overlap_duration=15)
    segments = [{"start": s.start, "text": s.text} for s in result.sentences]
    print(f"Transcribed {len(segments)} segments (parakeet-mlx)")
except ImportError:
    import os
    os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    import torch
    import whisperx
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    print("Loading whisperx turbo...")
    model = whisperx.load_model("turbo", device, compute_type=compute_type)
    audio = whisperx.load_audio(str(audio_path))
    print("Transcribing...")
    result = model.transcribe(audio, batch_size=16)
    segments = [{"start": s["start"], "text": s["text"].strip()} for s in result["segments"]]
    print(f"Transcribed {len(segments)} segments (whisperx)")


Loading parakeet-mlx...


Transcribing (chunked for long audio)...


Transcribed 2039 segments (parakeet-mlx)


**What do we want to know from the meeting?** Maybe agenda items, votes, public comments, and some vague general details.


In [ ]:
class AgendaItem(BaseModel):
    summary: str = Field(description="One-sentence summary of what was discussed")
    timestamp: str = Field(description="Approximate timestamp, MM:SS")

class Vote(BaseModel):
    item: str = Field(description="What was voted on")
    result: str = Field(description="Passed/failed/tabled and vote count if stated")
    timestamp: str = Field(description="MM:SS")

class PublicComment(BaseModel):
    speaker: str = Field(description="Name of commenter, if stated")
    topic: str = Field(description="What they spoke about")
    stance: str = Field(description="For/against/neutral")
    timestamp: str = Field(description="MM:SS")

class MeetingMinutes(BaseModel):
    title: str = Field(description="Short title for the meeting")
    overall_summary: str = Field(description="2-3 sentence summary of the meeting")
    agenda_items: list[AgendaItem] = Field(description="Major agenda items discussed")
    votes: list[Vote] = Field(description="Votes taken and their outcomes")
    public_comments: list[PublicComment] = Field(description="Public comments from residents")
    notable_quotes: list[str] = Field(description="Direct quotes that stood out, with speaker name")


Now we can send the transcript to Gemini for all the details.


In [ ]:
transcript_with_timestamps = "\n".join(
    f"[{int(seg['start']//60):02d}:{int(seg['start']%60):02d}] {seg['text']}"
    for seg in segments
)

print("Sending transcript to Gemini for summarization...")
agent = Agent(MODEL, output_type=MeetingMinutes)
summary = agent.run_sync(
    f"Summarize this city council meeting transcript. Extract agenda items, "
    f"votes and outcomes, public comments, and notable quotes. "
    f"Use timestamps from the transcript.\n\n"
    f"{transcript_with_timestamps}"
)


Sending transcript to Gemini for summarization...


And here's our structured meeting minutes.


In [ ]:
minutes = summary.output
print(f"# {minutes.title}\n")
print(f"{minutes.overall_summary}\n")

print("## Agenda Items")
for item in minutes.agenda_items:
    print(f"  [{item.timestamp}] {item.summary}")

print("\n## Votes")
for v in minutes.votes:
    print(f"  [{v.timestamp}] {v.item} — {v.result}")

print("\n## Public Comments")
for pc in minutes.public_comments:
    print(f"  [{pc.timestamp}] {pc.speaker}: {pc.topic} ({pc.stance})")

print("\n## Notable Quotes")
for q in minutes.notable_quotes:
    print(f"  \"{q}\"")


# Lewiston City Council Meeting Summary

The Lewiston City Council meeting covered a range of topics, including the acceptance of previous meeting minutes, recognition of outgoing councilors, and a proclamation for the homeless shelter committee. Major discussions and votes centered around the rejection of an AI data center development, the approval of a housing development TIF amendment, and the condemnation of a dangerous building. The council also addressed a controversial resolve to review the qualifications of a councilor-elect and received an update on various housing projects.

## Agenda Items
  [10:05] Acceptance of minutes from the December 2nd council meeting.
  [10:51] Presentation of plaques to outgoing Councilors Aaron M. Sowell and Timothy J. Gallant for their dedicated service.
  [13:35] Proclamation recognizing the City Council Ad Hoc Committee on Establishing a Homeless Shelter for its work in addressing homelessness.
  [30:20] Passage of six consent agenda items inclu

We could have done this other ways: pass the entire thing to Gemini, use local models for the summarization/extraction, all kinds of options. But it's a good starting point!

> I also tried [to do the same thing with Google Opal](https://developers.google.com/opal), and it just *hallucinated every single aspect of the meeting*. I don't even think [the end result](https://opal.google/edit/1k6P7domgII6AZ6edCprG34Bn9v0dh25e) was working from a transcript, despite the fact that Google owns YouTube. Weird!

Trust but verify!

## Cost

Before you run a big batch: how much will it cost? Images × tokens × price = receipt. I would *not* trust the below, but it'll give you a general idea of what processing 500 images might look like, and the variation across different models.


**`recipes/cost.py`** — Estimate API cost for a batch of images. No API key needed.


In [ ]:
# USD per million tokens (Feb 2026)
PRICE_TABLE = {
    "gpt-4o-mini":       (0.15, 0.60),
    "gpt-4o":            (2.50, 10.00),
    "gpt-5-nano":        (0.05, 0.20),
    "gemini-2.5-flash":  (0.10, 0.40),
    "gemini-2.5-pro":    (1.25, 5.00),
    "claude-3-5-haiku":  (0.80, 4.00),
    "claude-3-5-sonnet": (3.00, 15.00),
    "ollama":            (0.00, 0.00),
}

NUM_IMAGES = 500
INPUT_TOKENS_PER_IMAGE = 1000
OUTPUT_TOKENS_PER_IMAGE = 200

print(f"Images:      {NUM_IMAGES:,}")

for model in PRICE_TABLE.keys():
    input_price, output_price = PRICE_TABLE[model]

    # --- Calculate ---
    total_input = NUM_IMAGES * INPUT_TOKENS_PER_IMAGE
    total_output = NUM_IMAGES * OUTPUT_TOKENS_PER_IMAGE
    input_cost = (total_input / 1_000_000) * input_price
    output_cost = (total_output / 1_000_000) * output_price
    total_cost = input_cost + output_cost

    print(f"Model:       {model}")
    print(f"Input cost:  ${input_cost:.4f}  ({total_input:,} tokens @ ${input_price}/M)")
    print(f"Output cost: ${output_cost:.4f}  ({total_output:,} tokens @ ${output_price}/M)")
    print(f"TOTAL:       ${total_cost:.4f}")
    print("-" * 40)


Images:      500
Model:       gpt-4o-mini
Input cost:  $0.0750  (500,000 tokens @ $0.15/M)
Output cost: $0.0600  (100,000 tokens @ $0.6/M)
TOTAL:       $0.1350
----------------------------------------
Model:       gpt-4o
Input cost:  $1.2500  (500,000 tokens @ $2.5/M)
Output cost: $1.0000  (100,000 tokens @ $10.0/M)
TOTAL:       $2.2500
----------------------------------------
Model:       gpt-5-nano
Input cost:  $0.0250  (500,000 tokens @ $0.05/M)
Output cost: $0.0200  (100,000 tokens @ $0.2/M)
TOTAL:       $0.0450
----------------------------------------
Model:       gemini-2.5-flash
Input cost:  $0.0500  (500,000 tokens @ $0.1/M)
Output cost: $0.0400  (100,000 tokens @ $0.4/M)
TOTAL:       $0.0900
----------------------------------------
Model:       gemini-2.5-pro
Input cost:  $0.6250  (500,000 tokens @ $1.25/M)
Output cost: $0.5000  (100,000 tokens @ $5.0/M)
TOTAL:       $1.1250
----------------------------------------
Model:       claude-3-5-haiku
Input cost:  $0.4000  (500,000 t